# Post-processing of NWT Latin Hypercube

In [ ]:
import os
import dask
import xarray as xr
import numpy as np
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import above_library.processing as proc

import importlib

In [ ]:
# Setup PBSCluster
cluster = PBSCluster(
    cores=1,                                                   # The number of cores you want
    memory='25GB',                                             # Amount of memory
    processes=1,                                               # How many processes
    queue='casper',                                            # The type of queue to utilize
    local_directory='/glade/work/afoster',                     # Use your local directory
    resource_spec='select=1:ncpus=1:mem=25GB',                 # Specify resources
    log_directory='/glade/derecho/scratch/afoster/dask_logs',  # log directory
    account='P93300606',                                       # Input your project ID here
    walltime='08:00:00',                                       # Amount of wall time
    interface='ext')

In [ ]:
cluster.scale(80)
dask.config.set({
    'distributed.dashboard.link': 'https://jupyterhub.hpc.ucar.edu/stable/user/{USER}/proxy/{port}/status'
})
client = Client(cluster)
client

In [4]:
# client.shutdown()

In [5]:
# directories
top_dir = '/glade/work/afoster/ABoVE'
postp_dir = '/glade/derecho/scratch/afoster/ABOVE_black_spruce_outputs'
archive_dir = '/glade/derecho/scratch/afoster/black_spruce/archive'

all_keys = np.arange(1, 601)

In [6]:
# we will only read in these variables
hist_vars = ['FATES_GPP', 'FATES_NPLANT_SZ', 'FATES_VEGC',
             'FATES_LAI', 'FATES_BASALAREA_SZ',
             'FATES_BA_WEIGHTED_HEIGHT']


# ensemble tag (for creating ensemble variable)
ensemble_tag = 'NWT_singlept_ZF14-207_la_per_sa_NWT_LH_'


# must have this many years in dataset to be considered "complete"
num_years = 100

In [7]:
# run through and post-process all ensemble members
# script will check to see if we have already written out the file and will skip if so
keys_finished = proc.post_process_ensemble(archive_dir, postp_dir,
                                           ensemble_tag, hist_vars, num_years)

Member 1 exists, skipping...
Member 2 exists, skipping...
Member 3 exists, skipping...
Member 4 exists, skipping...
Member 5 exists, skipping...
Member 6 exists, skipping...
Member 7 exists, skipping...
Member 8 exists, skipping...
Member 9 exists, skipping...
Member 10 exists, skipping...
Member 11 exists, skipping...
Member 12 exists, skipping...
Member 13 exists, skipping...
Member 14 exists, skipping...
Member 15 exists, skipping...
Member 16 exists, skipping...
Member 17 exists, skipping...
Member 18 exists, skipping...
Member 19 exists, skipping...
Member 20 exists, skipping...
Member 21 exists, skipping...
Member 22 exists, skipping...
Member 23 exists, skipping...
Member 24 exists, skipping...
Member 25 exists, skipping...
Member 26 exists, skipping...
Member 27 exists, skipping...
Member 28 exists, skipping...
Member 29 exists, skipping...
Member 30 exists, skipping...
Member 31 exists, skipping...
Member 32 exists, skipping...
Member 33 exists, skipping...
Member 34 exists, s

In [8]:
missing_keys = proc.find_missing_keys(all_keys, keys_finished)

In [9]:
len(missing_keys)

0

In [ ]:
with open(os.path.join(top_dir, 'missing_keys.txt'), 'w') as f:
    for key in missing_keys:
        f.write(f"{key}\n")

In [10]:
model_dat = proc.get_ensemble(postp_dir).isel(lndgrid=0)

In [12]:
model_dat.to_netcdf('/glade/work/afoster/ABoVE/hist_outputs/black_spruce_ens.nc')